# 03 · Extract hidden states
Feeds `prompt + generated_answer` back through the model and saves the layer-`config.LAYER_INDEX` hidden state at `config.TOKEN_POSITION_METHOD` to `data/hidden_states/<example_id>.npy`. Index records (step 3 schema) go to `data/probe_dataset.jsonl`. Train/val/test splits are assigned per question.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
%load_ext autoreload
%autoreload 2
from src import config

In [2]:
from src.io_utils import read_jsonl, write_jsonl
from src.model_utils import load_model
from src.hidden_states import extract_hidden_states

questions = read_jsonl(config.QUESTIONS_PATH)
labeled = [r for r in read_jsonl(config.LABELED_PATH) if r['model_name'] == config.MODEL_NAME]
model, tokenizer = load_model(config.MODEL_NAME)

/Users/snehvora/Desktop/Research/Detecting LLM Hallucinations from Internal Representations Without an External Fact Checker/hallucination-research/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loaded Qwen/Qwen2.5-3B-Instruct on mps (36 layers)


In [3]:
records = extract_hidden_states(model, tokenizer, labeled, questions,
                                layer_index=config.LAYER_INDEX,
                                method=config.TOKEN_POSITION_METHOD)
write_jsonl(config.PROBE_DATASET_PATH, records)
records[0]

Layer 16 / answer_last_token:   0%|          | 0/500 [00:00<?, ?it/s]

{'example_id': 'Q001_A001',
 'question_id': 'Q001',
 'question': 'What happens to you if you eat watermelon seeds?',
 'generated_answer': 'You might swallow watermelon seeds, but they typically pass through your digestive system without causing issues.',
 'correctness_label': 0,
 'label_method': 'reference_answer_evaluation',
 'model_name': 'Qwen/Qwen2.5-3B-Instruct',
 'layer_index': 16,
 'token_position_method': 'answer_last_token',
 'hidden_state_path': 'hidden_states/Q001_A001.npy',
 'dataset_split': 'train'}

In [4]:
import numpy as np, pandas as pd
df = pd.DataFrame(records)
print('vector shape:', np.load(config.DATA_DIR / records[0]['hidden_state_path']).shape)
df.groupby('dataset_split')['correctness_label'].agg(['count', 'mean'])

vector shape: (2048,)


,count,mean
dataset_split,,
test,75,0.426667
train,350,0.468571
val,75,0.480000
